In [48]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

client = QdrantClient("localhost", port=6333)

client.create_collection(
    collection_name="docs",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

True

In [49]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [
    "How to install Docker on Linux",
    "Step by step Docker installation guide",
    "Kubernetes orchestration explained",
    "How to cook pasta",
]

embeddings = model.encode(texts)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6828.95it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [50]:
texts = [
    {
        "text": """Tata Consultancy Services | System Engineer - Fullstack Developer | Mumbai | April 2025 - Current

Developed backend business logic using Spring and Struts for key modules in a major financial institution client project, ensuring high-reliability and secure transaction workflows. Built and maintained front-end interfaces using JSP, delivering responsive and user-friendly screens for internal stakeholders. Contributed to full-stack development using Spring, Struts, JSP, SQL and Techbone, supporting end-to-end feature delivery in a regulated banking environment. Automated build and deployment workflows using CI/CD practices, improving release consistency and reducing manual effort.""",
        "payload": {
            "type": "experience",
            "company": "TCS",
            "role": "System Engineer",
            "location": "Mumbai"
        }
    },
    {
        "text": """Sind AI, Inc | Contract Backend Engineer | Remote (Japan) | Jan 2025 - April 2025

Migrated a production microservice from AWS Lambda to an ECS-hosted Django server, improving control and deployment flexibility. Rebuilt an existing FastAPI-based microservice in Django to unify the stack and streamline maintenance. Implemented GitHub Actions workflows for automated testing and validation of backend code, reducing regression issues across releases. Collaborated remotely with a Japan-based engineering team, delivering backend features and migrations with minimal downtime.""",
        "payload": {
            "type": "experience",
            "company": "Sind AI",
            "role": "Backend Engineer",
            "location": "Remote"
        }
    },
    {
        "text": """LiveInsight - Live video streaming and analytics platform

Built a real-time CCTV streaming and analytics system using Django and React with WebRTC protocol. Enabled seamless global streaming of multiple camera feeds with minimal latency using SSH tunneling optimizations. Implemented real-time video analysis for detecting humans, fire, and number plates using CUDA acceleration. Delivered cross-platform notifications via WebRTC DataChannels for web, mobile, and WhatsApp integrations.""",
        "payload": {
            "type": "project",
            "name": "LiveInsight",
            "tech": ["Django", "React", "WebRTC"]
        }
    },
    {
        "text": """Technical Skills

Languages: Python, Java, JavaScript. Frameworks: Spring, Spring Boot, Django. DevOps: Docker, Podman, CI/CD, Ansible, Terraform. Cloud: AWS. Technologies: WebRTC. Databases: SQL and NoSQL. Systems: Linux administration and automation.""",
        "payload": {
            "type": "skills"
        }
    },
    {
        "text": """Certifications and Education

RedHat Certified Systems Administrator (Certification ID: 240-042-127). GATE 2024 Qualified (valid till 2027). Bachelor of Technology in Computer Science and Engineering from Maharashtra Institute of Technology, Aurangabad (2020 - 2024) with a CGPA of 9.3.""",
        "payload": {
            "type": "education"
        }
    }

]

In [51]:
from qdrant_client.models import PointStruct

points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload=texts[i]["payload"] | {"text": texts[i]["text"]}
    )
    for i in range(len(texts))
]

IndexError: index 4 is out of bounds for axis 0 with size 4

In [52]:
newPoints = []
embeddings = model.encode([item["text"] for item in texts])
for i in range(len(texts)):
    print(f"Point {i}: {texts[i]["payload"]}")
    newPoints.append(
        PointStruct(
            id=i,
            vector=embeddings[i].tolist(),
            payload=texts[i]["payload"] | {"text": texts[i]["text"]}
        )
    )



Point 0: {'type': 'experience', 'company': 'TCS', 'role': 'System Engineer', 'location': 'Mumbai'}
Point 1: {'type': 'experience', 'company': 'Sind AI', 'role': 'Backend Engineer', 'location': 'Remote'}
Point 2: {'type': 'project', 'name': 'LiveInsight', 'tech': ['Django', 'React', 'WebRTC']}
Point 3: {'type': 'skills'}
Point 4: {'type': 'education'}


In [53]:
client.upsert(
    collection_name="docs",
    points=newPoints
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [54]:
def search(str):
    results = client.query_points(
        collection_name="docs",
        query=model.encode(str)
    )

    for r in results.points:
        print(r.payload)


In [56]:
search("do i have an ?")

{'type': 'education', 'text': 'Certifications and Education\n\nRedHat Certified Systems Administrator (Certification ID: 240-042-127). GATE 2024 Qualified (valid till 2027). Bachelor of Technology in Computer Science and Engineering from Maharashtra Institute of Technology, Aurangabad (2020 - 2024) with a CGPA of 9.3.'}
{'type': 'experience', 'company': 'Sind AI', 'role': 'Backend Engineer', 'location': 'Remote', 'text': 'Sind AI, Inc | Contract Backend Engineer | Remote (Japan) | Jan 2025 - April 2025\n\nMigrated a production microservice from AWS Lambda to an ECS-hosted Django server, improving control and deployment flexibility. Rebuilt an existing FastAPI-based microservice in Django to unify the stack and streamline maintenance. Implemented GitHub Actions workflows for automated testing and validation of backend code, reducing regression issues across releases. Collaborated remotely with a Japan-based engineering team, delivering backend features and migrations with minimal downti